# Classificação — Comparação de Modelos (Validação Cruzada)

**Objetivo:** comparar LogisticRegression, RandomForest e XGBoost de forma justa usando validação cruzada estratificada no conjunto de treino, com métricas apropriadas para dados desbalanceados (F1 e ROC-AUC, não apenas accuracy).

Continua a partir de `02_Preprocessamento_Baseline.ipynb` — mesmo split (mesma seed) e mesmo pré-processamento, definidos em `src/data.py`.

## 1. Imports

In [ ]:
import sys
sys.path.append('..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

from src.data import RANDOM_STATE, TEST_SIZE, carregar_dados, construir_preprocessador, separar_x_y

sns.set_theme(style='whitegrid')

## 2. Carregar dados e split (mesmo critério do notebook anterior)

In [ ]:
df = carregar_dados()
X, y = separar_x_y(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# Peso da classe positiva para o XGBoost (nao tem class_weight nativo)
escala_pos = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight:', round(escala_pos, 2))

## 3. Modelos candidatos

In [ ]:
modelos = {
    'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=300, scale_pos_weight=escala_pos, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1),
}

## 4. Validação cruzada estratificada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
metricas = ['accuracy', 'f1', 'roc_auc']

resultados = []
for nome, modelo in modelos.items():
    pipeline = Pipeline([
        ('preprocessador', construir_preprocessador()),
        ('modelo', modelo),
    ])
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=metricas, n_jobs=-1)
    for metrica in metricas:
        for valor in scores[f'test_{metrica}']:
            resultados.append({'modelo': nome, 'metrica': metrica, 'valor': valor})

resultados = pd.DataFrame(resultados)
resultados.groupby(['modelo', 'metrica'])['valor'].agg(['mean', 'std']).round(4)

## 5. Comparação visual

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=resultados, x='metrica', y='valor', hue='modelo', errorbar='sd')
plt.title('Comparação de modelos — validação cruzada (5 folds)')
plt.ylim(0, 1)
plt.savefig('../reports/comparacao_modelos_cv.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Avaliação do melhor modelo no conjunto de teste

Ajuste `MELHOR_MODELO` conforme o resultado da comparação acima (maior F1/ROC-AUC médios).

In [ ]:
MELHOR_MODELO = 'XGBoost'

pipeline_final = Pipeline([
    ('preprocessador', construir_preprocessador()),
    ('modelo', modelos[MELHOR_MODELO]),
])
pipeline_final.fit(X_train, y_train)

pred = pipeline_final.predict(X_test)
proba = pipeline_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap='Blues', ax=eixos[0])
eixos[0].set_title(f'Matriz de confusão — {MELHOR_MODELO}')
RocCurveDisplay.from_predictions(y_test, proba, ax=eixos[1])
eixos[1].set_title('Curva ROC')
plt.tight_layout()
plt.savefig('../reports/melhor_modelo_avaliacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Salvar modelo

In [ ]:
joblib.dump(pipeline_final, f'../models/{MELHOR_MODELO.lower()}_cv.joblib')

## 8. Próximos passos
- Otimização bayesiana de hiperparâmetros com Optuna → `04_Otimizacao_Optuna.ipynb`